# Phase 1.5 — 1a PASS 견고화 + top-k (K≈4)
Plan: `~/.claude/plans/snuggly-gathering-rossum.md`. 코드는 `phase1_5/` (Drive 동기화 필요).
- **Phase 2**: A.1 × seed{0,1,2} × use_best_val{True,False} → PASS 견고성(seed/final-K/ReClor/ceiling).
- **Phase 3**: routing='topk' (K_active≡4 by construction) → 영역-정합 재측정.
각 run 결과 = `operation_gate`(regex 축, lower-bound) + `operation_gate_reclor` + `operation_ceiling_raw`.

In [ ]:
# --- setup ---
import os, sys, glob, json
from pathlib import Path
import torch
BASE = Path('/content/drive/MyDrive/sideproject')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from phase1_5.data import MCCorpusConfig
cfg = MCCorpusConfig(cache_root=str(BASE / 'out' / 'phase1_5' / 'cache'))
print('BASE', BASE, '| device', device)

def show_op(res, label=''):
    d = json.load(open(res)) if isinstance(res, str) else res
    for key in ['operation_gate', 'operation_gate_reclor', 'operation_ceiling_raw']:
        g = d.get(key) or {}
        if not g:
            continue
        c = g.get('consistency') or {}
        print(f'[{label}|{key}] verdict={g.get("verdict")} adj_op={g.get("adj_operation")} '
              f'thr={g.get("threshold")} purity={c.get("op_purity")}/{c.get("op_chance")} '
              f'beats_topic={c.get("op_beats_topic")} n={g.get("n_operation_examples")}')

In [ ]:
# --- Phase 2: replication (seed {0,1,2} x use_best_val {True, False}) ---
# ~6 run x 24min. skip_if_exists=True → 중단/재개 OK. 급하면 seeds=[0,1].
from phase1_5.ablations import AblationRow, run_ablation_row
from phase1_5.data import MODE_Q_ONLY
from phase1_5.train import TrainConfig
from phase1_5.model import MOD_KG_HYPERNET

for seed in [0, 1, 2]:
    for ubv in [True, False]:
        row = AblationRow(row_id=f'R_s{seed}_bv{int(ubv)}', name=f'A.1 s{seed} bv{ubv}',
                          encoding_mode=MODE_Q_ONLY, k_routed=64, k_active_target=4.0,
                          modulation=MOD_KG_HYPERNET, lb_strategy='aux_free')
        tc = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, use_best_val=ubv, seed=seed)
        r = run_ablation_row(row, corpus_cfg=cfg, train_cfg=tc,
                             out_dir=str(BASE / 'out' / 'phase1_5' / 'replication'),
                             device=device, seed=seed, batch_size=64,
                             skip_if_exists=True, progress=False)
        show_op(r, f's{seed}_bv{int(ubv)}')

In [ ]:
# --- Phase 3: top-k routing (K_active = 4 by construction, ADR 0002) ---
# aux_free LB steers selection; L1 inert (softmax sum=1); z-loss for stability.
from phase1_5.ablations import AblationRow, run_ablation_row
from phase1_5.data import MODE_Q_ONLY
from phase1_5.train import TrainConfig
from phase1_5.model import MOD_KG_HYPERNET

row = AblationRow(row_id='T_topk4', name='Q_only topk K=4', encoding_mode=MODE_Q_ONLY,
                  k_routed=64, k_active_target=4.0, modulation=MOD_KG_HYPERNET,
                  lb_strategy='aux_free', routing='topk')
tc = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, lam_z=1e-3, use_best_val=False, seed=0)
res_topk = run_ablation_row(row, corpus_cfg=cfg, train_cfg=tc,
                            out_dir=str(BASE / 'out' / 'phase1_5' / 'topk'),
                            device=device, seed=0, batch_size=64,
                            skip_if_exists=False, progress=True)
show_op(res_topk, 'topk4')
print('K_active(last ep):', (res_topk.get('history') or [{}])[-1].get('k_active_mean'),
      '| val_acc:', (res_topk.get('history') or [{}])[-1].get('val_mc_acc'))

In [ ]:
# --- read all saved results (replication + topk) ---
for sub in ['replication', 'topk']:
    for p in sorted(glob.glob(str(BASE / 'out' / 'phase1_5' / sub / 'row_*.json'))):
        show_op(p, Path(p).stem)